In [4]:
from pathlib import Path
import cv2
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import torchvision.models as models

import albumentations as A
from albumentations.pytorch import ToTensorV2

In [5]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [6]:
train_transform = A.Compose([

    A.Resize(224,224),

    A.HorizontalFlip(p=0.5),

    A.RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2,
        p=0.5
    ),

    A.Rotate(limit=15, p=0.5),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])

val_transform = A.Compose([

    A.Resize(224,224),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])

In [7]:
class CattleDataset(Dataset):

    def __init__(self, root_dir, transform=None):

        self.root_dir = Path(root_dir)
        self.transform = transform

        self.image_paths = []
        self.labels = []

        self.class_names = sorted([
            folder.name
            for folder in self.root_dir.iterdir()
            if folder.is_dir()
        ])

        self.class_to_idx = {
            cls_name: idx
            for idx, cls_name in enumerate(self.class_names)
        }

        for cls_name in self.class_names:

            cls_folder = self.root_dir / cls_name

            for img_path in cls_folder.glob("*"):

                self.image_paths.append(img_path)
                self.labels.append(
                    self.class_to_idx[cls_name]
                )

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        img_path = self.image_paths[idx]

        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        label = self.labels[idx]

        if self.transform:
            image = self.transform(image=image)["image"]

        return image, label

In [8]:
train_dataset = CattleDataset(
    root_dir="../datasets/identity_split/train",
    transform=train_transform
)

val_dataset = CattleDataset(
    root_dir="../datasets/identity_split/val",
    transform=val_transform
)

In [9]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

In [10]:
import math

class ArcFace(nn.Module):

    def __init__(self,
                 in_features,
                 out_features,
                 s=30.0,
                 m=0.50):

        super().__init__()

        self.s = s
        self.m = m

        self.weight = nn.Parameter(
            torch.FloatTensor(
                out_features,
                in_features
            )
        )

        nn.init.xavier_uniform_(
            self.weight
        )

    def forward(
        self,
        embeddings,
        labels
    ):

        embeddings = F.normalize(
            embeddings
        )

        weights = F.normalize(
            self.weight
        )

        cosine = F.linear(
            embeddings,
            weights
        )

        theta = torch.acos(
            torch.clamp(
                cosine,
                -1 + 1e-7,
                1 - 1e-7
            )
        )

        target_logits = torch.cos(
            theta + self.m
        )

        one_hot = F.one_hot(
            labels,
            num_classes=weights.shape[0]
        ).float()

        logits = (
            one_hot * target_logits
            +
            (1.0 - one_hot) * cosine
        )

        logits *= self.s

        return logits

In [11]:
class ArcFaceResNet50(nn.Module):

    def __init__(self,
                 num_classes):

        super().__init__()

        backbone = models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT
        )

        in_features = (
            backbone.fc.in_features
        )

        backbone.fc = nn.Identity()

        self.backbone = backbone

        self.embedding = nn.Linear(
            in_features,
            512
        )

        self.arcface = ArcFace(
            512,
            num_classes
        )

    def forward(
        self,
        images,
        labels=None
    ):

        features = self.backbone(
            images
        )

        embeddings = self.embedding(
            features
        )

        embeddings = F.normalize(
            embeddings
        )

        if labels is None:
            return embeddings

        logits = self.arcface(
            embeddings,
            labels
        )

        return logits, embeddings

In [12]:
num_classes = len(
    train_dataset.class_names
)

model = ArcFaceResNet50(
    num_classes=num_classes
)

model = model.to(device)

print("Classes:", num_classes)

Classes: 84


In [13]:
model.load_state_dict(
    torch.load(
        "../models/cattle_arcface.pth",
        weights_only=True
    )
)

model.eval()

print("ArcFace model loaded.")

ArcFace model loaded.


In [14]:
import faiss
import numpy as np

print("FAISS Version:", faiss.__version__)

FAISS Version: 1.14.2


In [15]:
embedding_database = []
label_database = []

with torch.no_grad():

    for images, labels in train_loader:

        images = images.to(device)

        embeddings = model(images)

        embedding_database.append(
            embeddings.cpu()
        )

        label_database.append(
            labels
        )

embedding_database = torch.cat(
    embedding_database
)

label_database = torch.cat(
    label_database
)

print(embedding_database.shape)
print(label_database.shape)

torch.Size([838, 512])
torch.Size([838])


In [16]:
embedding_np = embedding_database.numpy().astype(
    np.float32
)

labels_np = label_database.numpy()

print(embedding_np.shape)

(838, 512)


In [17]:
dimension = 512

index = faiss.IndexFlatIP(
    dimension
)

faiss.normalize_L2(
    embedding_np
)

index.add(
    embedding_np
)

print(
    "Vectors in Index:",
    index.ntotal
)

Vectors in Index: 838


In [18]:
import numpy as np

np.save(
    "../models/faiss_embeddings.npy",
    embedding_database.numpy()
)

np.save(
    "../models/faiss_labels.npy",
    label_database.numpy()
)

print("FAISS database saved.")

FAISS database saved.


In [19]:
from pathlib import Path

print(
    list(Path("../models").glob("*.npy"))
)

[WindowsPath('../models/faiss_embeddings.npy'), WindowsPath('../models/faiss_labels.npy')]


In [16]:
def identify_cow_faiss(
    query_embedding,
    index,
    labels_np,
    k=1
):

    query = (
        query_embedding
        .cpu()
        .numpy()
        .astype(np.float32)
        .reshape(1, -1)
    )

    faiss.normalize_L2(
        query
    )

    distances, indices = index.search(
        query,
        k
    )

    predicted_label = (
        labels_np[
            indices[0][0]
        ]
    )

    similarity = (
        distances[0][0]
    )

    return (
        predicted_label,
        similarity
    )

In [17]:
query_image, query_label = (
    train_dataset[50]
)

query_tensor = (
    query_image
    .unsqueeze(0)
    .to(device)
)

with torch.no_grad():

    query_embedding = model(
        query_tensor
    )

predicted_label, similarity = (
    identify_cow_faiss(
        query_embedding.squeeze(0),
        index,
        labels_np
    )
)

print(
    "Actual Label:",
    query_label
)

print(
    "Predicted Label:",
    predicted_label
)

print(
    "Similarity:",
    similarity
)

Actual Label: 5
Predicted Label: 5
Similarity: 0.94835126


In [3]:
for name in dir():
    if "embed" in name.lower():
        print(name)

for name in dir():
    if "label" in name.lower():
        print(name)

In [21]:
import torch.nn.functional as F

i, j = 0, 324

sim = F.cosine_similarity(
    embedding_database[i].unsqueeze(0),
    embedding_database[j].unsqueeze(0)
)

print("Similarity:", sim.item())

Similarity: 0.05435827374458313


In [22]:
for idx in range(20):
    print(idx, label_database[idx].item())

0 80
1 43
2 82
3 80
4 9
5 83
6 23
7 50
8 74
9 12
10 25
11 59
12 63
13 32
14 57
15 80
16 55
17 6
18 39
19 80


In [23]:
for idx in range(300,340):
    print(idx, label_database[idx].item())

300 35
301 78
302 5
303 45
304 73
305 0
306 26
307 10
308 81
309 9
310 15
311 35
312 14
313 18
314 59
315 14
316 16
317 22
318 14
319 30
320 5
321 22
322 66
323 6
324 18
325 82
326 8
327 70
328 33
329 14
330 50
331 50
332 25
333 83
334 10
335 21
336 79
337 75
338 32
339 16


In [24]:
import torch.nn.functional as F

i = 0
j = 3

sim = F.cosine_similarity(
    embedding_database[i].unsqueeze(0),
    embedding_database[j].unsqueeze(0)
)

print(sim.item())

0.959883451461792


In [25]:
print(len(train_dataset))
print(len(val_dataset))

838
341


In [ ]:
for path, label in train_dataset.samples[:10]:
    print(path, label)
    

AttributeError: 'CattleDataset' object has no attribute 'samples'

In [27]:
for path, label in val_dataset.samples[:10]:
    print(path, label)

AttributeError: 'CattleDataset' object has no attribute 'samples'

In [28]:
found = False

for path, label in train_dataset.samples:
    if "A_0.jpg" in path:
        print("TRAIN:", path, label)
        found = True

for path, label in val_dataset.samples:
    if "A_0.jpg" in path:
        print("VAL:", path, label)
        found = True

print("Found:", found)

AttributeError: 'CattleDataset' object has no attribute 'samples'

In [29]:
print(train_dataset.__dict__.keys())

dict_keys(['root_dir', 'transform', 'image_paths', 'labels', 'class_names', 'class_to_idx'])


In [30]:
print(len(train_dataset))
print(len(val_dataset))

838
341


In [31]:
img, label = train_dataset[0]

print(type(img))
print(label)

<class 'torch.Tensor'>
0


In [32]:
print(train_dataset.root_dir)

..\datasets\identity_split\train


In [34]:
print(train_dataset.class_names[:10])

['cattle_id_001', 'cattle_id_002', 'cattle_id_003', 'cattle_id_004', 'cattle_id_005', 'cattle_id_006', 'cattle_id_007', 'cattle_id_008', 'cattle_id_009', 'cattle_id_010']


In [35]:
from pathlib import Path

val_images = list(
    Path("../datasets/identity_split/val").rglob("*.jpg")
)

print(val_images[:10])

[WindowsPath('../datasets/identity_split/val/cattle_id_088/cattle_id_088_01.jpg'), WindowsPath('../datasets/identity_split/val/cattle_id_088/cattle_id_088_02.jpg'), WindowsPath('../datasets/identity_split/val/cattle_id_088/cattle_id_088_03.jpg'), WindowsPath('../datasets/identity_split/val/cattle_id_088/cattle_id_088_04.jpg'), WindowsPath('../datasets/identity_split/val/cattle_id_088/cattle_id_088_05.jpg'), WindowsPath('../datasets/identity_split/val/cattle_id_088/cattle_id_088_06.jpg'), WindowsPath('../datasets/identity_split/val/cattle_id_088/cattle_id_088_07.jpg'), WindowsPath('../datasets/identity_split/val/cattle_id_088/cattle_id_088_08.jpg'), WindowsPath('../datasets/identity_split/val/cattle_id_088/cattle_id_088_09.jpg'), WindowsPath('../datasets/identity_split/val/cattle_id_088/cattle_id_088_10.jpg')]
